# What can an accelerated weight drop resolve in a deep borehole on the San Andreas?

**One question. Four figures. Nothing else goes in this notebook.**

SAFOD main hole, June 2026 AWD survey: 989 GPS-timed weight drops over 24 h, recorded on a
cemented fiber (Lellouch et al. 2019's, 864 m) and a wireline fiber (~3.5 km).

The question has two halves:
1. **What does it measure?** &rarr; a P-wave velocity profile of the fault damage zone.
2. **What could it monitor?** &rarr; the repeatability floor sets the smallest detectable dv/v.

| Figure | Answers |
|---|---|
| 1 | Is there a coherent arrival at all, and how fast? |
| 2 | How does $V_P$ vary with depth? |
| 3 | How repeatable is it, and to what depth? |
| 4 | Is that good enough to monitor the San Andreas? |

Everything here reads already-computed stacks. No raw `.pb`/`.h5` access, no re-stacking.

**Standing caveats, stated once:**
- Depth is *distance along fiber* with an **uncalibrated absolute offset**. The cemented
  fiber's interrogator-to-wellhead lead-in is unknown; the wireline fiber's array starts
  3678 m along the fiber (`StartLocusIndex = 1800`). Profile *shape* is what is defensible.
- Lellouch et al. 2019 report the cemented fiber ends at **864 m** with a failed end loop,
  capping usable data at **800 m**. Whether it was repaired before 2026 is unknown — an
  ambient-noise test was inconclusive and an OTDR is needed. We truncate at 800 m.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt

FIG_DIR = '/home/groups/ettore88/nberrios/safod_das_git/notebooks/figures/awd_2026'
OUT_DIR = os.path.join(FIG_DIR, 'scec')
os.makedirs(OUT_DIR, exist_ok=True)

# --- acquisition constants, all read from the file metadata (not assumed) -----
# Sintela .pb acquisition_stats  |  OptaSense .h5 /Acquisition attributes
DX_CEM,  GL_CEM  = 1.26606202, 16.4588    # cemented fiber, m
DX_WIRE, GL_WIRE = 2.0419,     10.2095    # wireline fiber, m
FS = 1000.0        # both: 10 kHz ping rate decimated x10 -> 1 kHz output
PRE_S = 0.5        # pre-drop window in the stored stacks

# --- published bounds, cemented fiber (Lellouch et al. 2019, 10.1029/2019JB017533)
FIBER_END_M = 864.0
FAILURE_M   = 800.0
ALLUVIUM_M  = 75.0     # shallow section crosses alluvium; excluded below

# --- analysis window ----------------------------------------------------------
Z_MIN, Z_MAX = 130.0, 530.0   # top: alluvium + near-source. bottom: SNR=3 limit
BAND = (20.0, 50.0)

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.titlesize': 11, 'font.size': 9})
print('constants loaded')

In [ ]:
# Load the paired stacks and build one weighted record section.
# Weighting by drops-per-epoch means a burst that captured 25 drops counts more
# than one that captured 5, which is what an unweighted mean would get wrong.
d = np.load(os.path.join(FIG_DIR, 'epoch_stacks_paired.npz'))
n_common = d['n_common']
good = n_common > 0
w = n_common[good].astype(float)

cem = np.tensordot(w, d['nano_stacks'][good], axes=(0, 0)) / w.sum()
N_DROPS = int(w.sum())
i0 = int(PRE_S * FS)

def bandpass(x, lo, hi, fs=FS):
    return sosfiltfilt(butter(4, [lo, hi], btype='band', fs=fs, output='sos'), x, axis=-1)

cem_bp = bandpass(cem, *BAND)
z_cem = np.arange(cem.shape[0]) * DX_CEM
print(f'{good.sum()} epochs, {N_DROPS} drops, cemented fiber {cem.shape}')
print(f'fiber spans 0-{z_cem[-1]:.0f} m; truncating analysis at {FAILURE_M:.0f} m')

## Figure 1 &mdash; Is there a coherent arrival, and how fast?

Measured by **slant-stack semblance**, not by picking. Two attempts at first-break picking
locked onto the wrong arrival — first a late coda 1–3 s after the drop, then energy near
1100 m/s which in a fluid-filled borehole is a tube wave. Semblance makes no such choice:
it sums along every trial moveout and reports which velocities carry coherent energy, so
multiple arrivals appear as separate peaks.

In [ ]:
def semblance(sec, zrel, i_ref, vgrid, dtgrid, win_s=0.040, fs=FS):
    """Semblance over (velocity, intercept): coherent energy / total energy."""
    nw, nt = int(win_s * fs), sec.shape[1]
    out = np.zeros((vgrid.size, dtgrid.size))
    for iv, v in enumerate(vgrid):
        shifts = (zrel / v * fs).astype(int)
        for it, dt in enumerate(dtgrid):
            idx = i_ref + int(dt * fs) + shifts
            if idx.min() < 0 or idx.max() + nw >= nt:
                continue
            g = np.stack([sec[k, i:i + nw] for k, i in enumerate(idx)])
            den = zrel.size * np.sum(g ** 2)
            out[iv, it] = np.sum(np.sum(g, axis=0) ** 2) / den if den > 0 else 0.0
    return out

c0, c1 = int(Z_MIN / DX_CEM), int(Z_MAX / DX_CEM)
sec = cem_bp[c0:c1 + 1]
zabs = z_cem[c0:c1 + 1]
zrel = zabs - Z_MIN

V_GRID = np.arange(400.0, 6000.0, 25.0)
T0_GRID = np.arange(0.0, 0.25, 0.002)
sm = semblance(sec, zrel, i0, V_GRID, T0_GRID)

iv, it = np.unravel_index(np.argmax(sm), sm.shape)
V_MEAS, T0_MEAS, S_MEAS = V_GRID[iv], T0_GRID[it], sm[iv, it]
print(f'dominant moveout: {V_MEAS:.0f} m/s, t0 = {T0_MEAS*1e3:.1f} ms, semblance {S_MEAS:.3f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))

clip = np.percentile(np.abs(sec), 99)
ax[0].imshow(sec.T, aspect='auto', cmap='gray_r',
             extent=[zabs[0], zabs[-1], (sec.shape[1] - i0) / FS, -PRE_S],
             vmin=-clip, vmax=clip)
ax[0].plot(zabs, T0_MEAS + zrel / V_MEAS, 'C3', lw=1.6,
           label=f'{V_MEAS:.0f} m/s (measured)')
ax[0].set(ylim=(0.45, -0.05), xlabel='distance along fiber (m)',
          ylabel='time after drop (s)',
          title=f'A  Stacked AWD record, {N_DROPS} drops, {BAND[0]:.0f}–{BAND[1]:.0f} Hz')
ax[0].legend(loc='lower right')

im = ax[1].imshow(sm.T, aspect='auto', origin='lower', cmap='magma',
                  extent=[V_GRID[0], V_GRID[-1], 0, T0_GRID[-1] * 1e3])
ax[1].plot(V_MEAS, T0_MEAS * 1e3, 'co', ms=9, mfc='none', mew=2)
ax[1].set(xlabel='trial velocity (m/s)', ylabel='intercept $t_0$ (ms)',
          title=f'B  Semblance — peak {S_MEAS:.2f} at {V_MEAS:.0f} m/s')
ax[1].grid(False)
plt.colorbar(im, ax=ax[1], label='semblance')

fig.suptitle('Fig 1 — A coherent direct-P arrival is present and its velocity is measured',
             fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig1_arrival_and_velocity.png'), bbox_inches='tight')
plt.show()

## Figure 2 &mdash; How does $V_P$ vary with depth?

Same semblance in overlapping 100 m windows, reparameterised so the **interval** velocity is
what is solved for rather than trading off against a global intercept:
$t(z) = t_\mathrm{top} + (z - z_\mathrm{top})/V_\mathrm{int}$.

Peak semblance per window is plotted alongside — it falls from ~0.9 to ~0.5 with depth, so the
deep points are much less constrained than the shallow ones.

In [ ]:
WIN_M, STEP_M = 100.0, 25.0
V_INT_GRID = np.arange(1500.0, 5500.0, 25.0)
DT_GRID = np.arange(-0.040, 0.040, 0.002)

centers = np.arange(Z_MIN + WIN_M / 2, Z_MAX - WIN_M / 2 + 1, STEP_M)
vp, vz, vq = [], [], []
for zc in centers:
    zt, zb = zc - WIN_M / 2, zc + WIN_M / 2
    a, b = int(zt / DX_CEM), int(zb / DX_CEM)
    s = semblance(cem_bp[a:b + 1], np.arange(a, b + 1) * DX_CEM - zt,
                  i0 + int(zt / V_MEAS * FS), V_INT_GRID, DT_GRID)
    i, j = np.unravel_index(np.argmax(s), s.shape)
    if i in (0, V_INT_GRID.size - 1):
        continue                       # peak pinned to the grid edge is not a measurement
    vp.append(V_INT_GRID[i]); vz.append(zc); vq.append(s[i, j])

vp, vz, vq = np.array(vp), np.array(vz), np.array(vq)
print(f'interval Vp over {vz[0]:.0f}–{vz[-1]:.0f} m: median {np.median(vp):.0f} m/s '
      f'(range {vp.min():.0f}–{vp.max():.0f})')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 6), sharey=True,
                       gridspec_kw={'width_ratios': [2, 1]})
ax[0].plot(vp, vz, 'C0-o', lw=2, ms=5)
ax[0].axvline(V_MEAS, ls=':', c='k', lw=1.2, label=f'whole-window {V_MEAS:.0f} m/s')
ax[0].set(xlabel='interval $V_P$ (m/s)', ylabel='distance along fiber (m)',
          title='A  Active-source $V_P$, cemented fiber')
ax[0].invert_yaxis()
ax[0].legend()

ax[1].plot(vq, vz, 'C2-o', lw=1.6, ms=5)
ax[1].set(xlabel='peak semblance', title='B  Constraint')

fig.suptitle(f'Fig 2 — $V_P$ rises {vp[:3].mean():.0f} → {vp[-4:].mean():.0f} m/s '
             f'({WIN_M:.0f} m windows, no error bars — see note)', fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig2_vp_profile.png'), bbox_inches='tight')
plt.show()

> **Open item on Fig 2.** These points carry no uncertainty estimate. The total variation is
> ~&plusmn;5% and the semblance peak broadens as SNR falls, so the deep structure may not be
> resolved. The defensible statement today is *"$V_P \approx$ 2850–2900 m/s at 180–230 m,
> rising to roughly 3100 m/s below 300 m."* A bootstrap over the 46 epochs would put real
> error bars on this and should be done before the poster prints.

## Figure 3 &mdash; How repeatable, and to what depth?

Loaded from the saved product, using its **stored** `scan_depth` axis (the scan channels are
subsampled across the fiber, so `arange(n)*dx` picks the wrong ones).

Truncated at the published 800 m failure depth. Untruncated, 19 of 126 scan channels sit past
it contributing CC &asymp; 0.02 — noise reported as measurement.

In [ ]:
r = np.load(os.path.join(FIG_DIR, 'within_burst_repeatability_shallow_20_50Hz.npz'))
zs = r['scan_depth']
keep = zs <= FAILURE_M
print(f'{(~keep).sum()} of {zs.size} scan channels dropped (past {FAILURE_M:.0f} m)')

metrics = [('cc_med', 'waveform CC', 0.7, None),
           ('nrms_med', 'NRMS (%)', 100.0, None),
           ('lag_scatter_med', 'timing scatter (ms)', 1.0, None)]

fig, ax = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
for k, (key, lab, thr, _) in enumerate(metrics):
    v = r[key][keep]
    ax[k].plot(v, zs[keep], 'C0-', lw=1.8)
    ax[k].axhspan(Z_MIN, Z_MAX, color='C2', alpha=0.10)
    ax[k].axvline(thr, ls='--', c='C3', lw=1.2)
    ax[k].set(xlabel=lab)
    if key == 'lag_scatter_med':
        ax[k].set_xscale('log')
ax[0].set(ylabel='distance along fiber (m)')
ax[0].invert_yaxis()

band = (zs >= Z_MIN) & (zs <= 400)
summary = {k: float(np.median(r[k][band])) for k, _, _, _ in metrics}
fig.suptitle(f"Fig 3 — Drop-to-drop repeatability. Over 130–400 m (shaded): "
             f"CC {summary['cc_med']:.2f}, NRMS {summary['nrms_med']:.0f}%, "
             f"timing {summary['lag_scatter_med']:.2f} ms", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig3_repeatability.png'), bbox_inches='tight')
plt.show()
print(summary)

## Figure 4 &mdash; Is that good enough to monitor the San Andreas?

The repeatability floor converts to a smallest-detectable velocity change. Compared against
the dv/v signals people actually chase at Parkfield.

> **The literature values below are placeholders and must be checked against the papers
> before this is shown.** Brenguier et al. (2008, *Science*) is the direct Parkfield
> comparison — same fault, same area — and its coseismic dv/v should be read off that paper
> rather than trusted from memory here.

In [ ]:
dv = np.load(os.path.join(FIG_DIR, 'shallow_direct_p_dvv_20_50Hz_locked.npz'))
EPS_FLOOR = float(dv['eps_scatter_clean'])          # achieved dv/v scatter
TARGET    = float(dv['defazio_target'])
print(f"achieved dv/v scatter {EPS_FLOOR*100:.3f}%  over "
      f"{float(dv['z_min_m']):.0f}–{float(dv['z_max_m']):.0f} m, CC > {float(dv['min_cc'])}")
print(f'target {TARGET*100:.3f}%  →  gap x{EPS_FLOOR/TARGET:.1f}')

# TODO(verify against sources before printing the poster)
SIGNALS = [
    ('Earth tides',                 1e-5,  1e-4),
    ('Seasonal / hydrologic',       3e-4,  2e-3),
    ('Coseismic drop, Parkfield',   3e-4,  3e-3),
    ('Large coseismic drop',        3e-3,  1e-2),
]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (name, lo, hi) in enumerate(SIGNALS):
    detect = lo >= EPS_FLOOR
    ax.barh(i, (hi - lo) * 100, left=lo * 100, height=0.55,
            color='C2' if detect else '0.75',
            edgecolor='k', lw=0.6)
    ax.text(hi * 100 * 1.25, i, 'detectable' if detect else 'below floor',
            va='center', fontsize=8, color='C2' if detect else '0.4')

ax.axvline(EPS_FLOOR * 100, color='C3', lw=2.2,
           label=f'achieved floor {EPS_FLOOR*100:.2f}%')
ax.axvline(TARGET * 100, color='C0', lw=1.6, ls='--',
           label=f'target {TARGET*100:.2f}%')
ax.set(xscale='log', yticks=range(len(SIGNALS)),
       yticklabels=[s[0] for s in SIGNALS],
       xlabel='|dv/v| (%)',
       title='Fig 4 — AWD + borehole DAS against dv/v signals at Parkfield')
ax.set_xlim(5e-4, 3)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig4_monitoring_floor.png'), bbox_inches='tight')
plt.show()

## The five numbers

Everything above, in one place. If a claim is not in this table, it is not supported.

In [ ]:
rows = [
    ('Direct-P velocity, 130–530 m', f'{V_MEAS:.0f} m/s', f'semblance {S_MEAS:.2f}'),
    ('Interval Vp range',            f'{vp.min():.0f}–{vp.max():.0f} m/s', 'no error bars yet'),
    ('Waveform CC, 130–400 m',       f"{summary['cc_med']:.2f}", 'drop-to-drop'),
    ('Timing scatter, 130–400 m',    f"{summary['lag_scatter_med']:.2f} ms", 'drop-to-drop'),
    ('dv/v floor',                   f'{EPS_FLOOR*100:.3f} %', f'x{EPS_FLOOR/TARGET:.1f} above target'),
]
w0 = max(len(r[0]) for r in rows)
print(f'{N_DROPS} weight drops, cemented fiber, {BAND[0]:.0f}–{BAND[1]:.0f} Hz\n')
for a, b, c in rows:
    print(f'{a:<{w0}}  {b:>16}   ({c})')

## What this notebook does not claim

- **Absolute depth.** Uncalibrated offset on both fibers. Needs an OTDR or a wellhead tap test.
- **That the cemented fiber is intact past 800 m.** Unresolved; we truncate rather than assume.
- **Anything comparing the two fibers' amplitudes.** Cemented is strain-rate, wireline is
  strain, and the gauge lengths differ (16.46 vs 10.21 m). Not attempted here.
- **Structure in $V_P(z)$ beyond the broad rise.** No error bars yet.

## Next, in order

1. Bootstrap over the 46 epochs for Fig 2 error bars.
2. Verify the Fig 4 literature values against the papers.
3. Get the OTDR — it closes the depth axis and the 800 m question at once.